# Demo 04: Power Iteration for Improved Accuracy

This demo shows how power iteration improves sketching accuracy.

Power iteration applies `(A'A)^q` to the random sketch, which amplifies
the dominant singular components. This is especially helpful when:
- The spectral gap is small
- High accuracy is needed
- The matrix has slowly decaying singular values

The demo tests a 2D grid of parameters:
- **extra_samples**: How much oversampling (more = better accuracy)
- **power_iter**: Number of power iterations (more = better accuracy)

Trade-off: More iterations/samples = better accuracy but more computation.

Try changing the **Configuration** parameters below to experiment!

In [24]:
# Configuration - Modify these to experiment

MATRIX_SIZE = (5000, 3000)    # (rows, columns)
TARGET_RANK = 50            # Number of singular values to compute
RANDOM_SEED = 42            # For reproducibility

# Power iteration settings - test grid of values
EXTRA_SAMPLES_LIST = [24, 18, 12, 6, 3]  # Oversampling values to test
POWER_ITER_LIST = [0, 1, 2, 3, 4, 8, 16]        # Power iteration counts to test

# Matrix type: 'structured' (clear spectral gap) or 'random' (no gap)
MATRIX_TYPE = 'structured'

In [25]:
import numpy as np
import time
import sys
sys.path.insert(0, '.')

from scipy import linalg
from librla import svd_sketch

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

m, n = MATRIX_SIZE
k = TARGET_RANK

print(f"Matrix: {m} x {n}")
print(f"Target rank: {k}")

# Create test matrix
if MATRIX_TYPE == 'structured':
    print("Matrix type: STRUCTURED")
    print("Singular values: logspace(0,-2,k) + logspace(-2,-10,n-k)")
    # Create matrix with decaying spectrum and clear gap at rank k
    U_full = linalg.orth(np.random.randn(m, m))
    V_full = linalg.orth(np.random.randn(n, n))
    s_true = np.concatenate([
        np.logspace(0, -2, k),      # Fast decay in first k singular values
        np.logspace(-2, -10, n-k)   # Slow decay after
    ])
    U = U_full[:, :n]
    A = U @ np.diag(s_true) @ V_full.T
else:
    print("Matrix type: RANDOM (no spectral gap)")
    A = np.random.randn(m, n)
    s_true = linalg.svd(A, compute_uv=False)

# Matrix properties
cond = s_true[0] / s_true[-1]
gap = s_true[k-1] / s_true[k] if k < len(s_true) else float('inf')

print(f"\nSpectral properties:")
print(f"   s[0]     = {s_true[0]:.6e} (largest)")
print(f"   s[{k-1}]   = {s_true[k-1]:.6e} (at target rank)")
print(f"   s[{k}]   = {s_true[k]:.6e} (first neglected)")
print(f"   s[{n-1}] = {s_true[n-1]:.6e} (smallest)")
print(f"   Condition number: {cond:.2e}")
print(f"   Spectral gap at k={k}: {gap:.1f}x")

Matrix: 5000 x 3000
Target rank: 50
Matrix type: STRUCTURED
Singular values: logspace(0,-2,k) + logspace(-2,-10,n-k)

Spectral properties:
   s[0]     = 1.000000e+00 (largest)
   s[49]   = 1.000000e-02 (at target rank)
   s[50]   = 1.000000e-02 (first neglected)
   s[2999] = 1.000000e-10 (smallest)
   Condition number: 1.00e+10
   Spectral gap at k=50: 1.0x


## Parameter Grid Test

- `power_iter=0` means no power iteration (baseline)
- Each power iteration costs 2 extra matrix-vector products
- `extra_samples` controls oversampling (`block_size = k + extra_samples`)

In [26]:
# Store results in 2D arrays
errors = np.zeros((len(EXTRA_SAMPLES_LIST), len(POWER_ITER_LIST)))
sval_errors = np.zeros((len(EXTRA_SAMPLES_LIST), len(POWER_ITER_LIST)))
s_ref = s_true[:k]

for idx, extra_samples in enumerate(EXTRA_SAMPLES_LIST):
    block_size = k + extra_samples
    print(f"\n--- extra_samples = {extra_samples} (block_size = {block_size}) ---")

    for jdx, power_iter in enumerate(POWER_ITER_LIST):
        t0 = time.perf_counter()
        U, s, Vh = svd_sketch(A, rtol=float(k),
                              power_iter=power_iter,
                              extra_samples=extra_samples)
        elapsed = time.perf_counter() - t0

        # Reconstruction error
        A_approx = U @ np.diag(s) @ Vh
        recon_err = np.linalg.norm(A - A_approx, 'fro') / np.linalg.norm(A, 'fro')
        errors[idx, jdx] = recon_err

        # Singular value accuracy
        sval_err = np.linalg.norm(s - s_ref) / np.linalg.norm(s_ref)
        sval_errors[idx, jdx] = sval_err

        print(f"   power_iter={power_iter}: err={recon_err:.2e}, sval_err={sval_err:.2e}, time={elapsed:.4f}s")


--- extra_samples = 24 (block_size = 74) ---
   power_iter=0: err=5.41e-02, sval_err=5.52e-03, time=0.0207s
   power_iter=1: err=3.73e-02, sval_err=5.00e-04, time=0.0310s
   power_iter=2: err=3.72e-02, sval_err=1.41e-04, time=0.0399s
   power_iter=3: err=3.71e-02, sval_err=7.02e-05, time=0.0504s
   power_iter=4: err=3.71e-02, sval_err=3.85e-05, time=0.0602s
   power_iter=8: err=3.71e-02, sval_err=3.19e-06, time=0.1007s
   power_iter=16: err=3.71e-02, sval_err=8.37e-09, time=0.1815s

--- extra_samples = 18 (block_size = 68) ---
   power_iter=0: err=5.73e-02, sval_err=6.59e-03, time=0.0184s
   power_iter=1: err=3.74e-02, sval_err=6.88e-04, time=0.0266s
   power_iter=2: err=3.72e-02, sval_err=2.32e-04, time=0.0361s
   power_iter=3: err=3.71e-02, sval_err=7.51e-05, time=0.0455s
   power_iter=4: err=3.71e-02, sval_err=3.40e-05, time=0.0546s
   power_iter=8: err=3.71e-02, sval_err=6.78e-06, time=0.0905s
   power_iter=16: err=3.71e-02, sval_err=5.54e-08, time=0.1630s

--- extra_samples = 12 

## Summary Tables

In [27]:
print("Reconstruction Error")
print("=" * 40)

# Header row
header = "extra_samples |"
for p in POWER_ITER_LIST:
    header += f"  iter={p}  |"
print(header)
print("-" * 14 + "+" + ("-" * 10 + "+") * len(POWER_ITER_LIST))

# Data rows
for idx, extra_samples in enumerate(EXTRA_SAMPLES_LIST):
    row = f"{extra_samples:>13} |"
    for jdx in range(len(POWER_ITER_LIST)):
        row += f" {errors[idx, jdx]:.2e} |"
    print(row)

print("\n")
print("Singular Value Error")
print("=" * 40)

# Header row
header = "extra_samples |"
for p in POWER_ITER_LIST:
    header += f"  iter={p}  |"
print(header)
print("-" * 14 + "+" + ("-" * 10 + "+") * len(POWER_ITER_LIST))

# Data rows
for idx, extra_samples in enumerate(EXTRA_SAMPLES_LIST):
    row = f"{extra_samples:>13} |"
    for jdx in range(len(POWER_ITER_LIST)):
        row += f" {sval_errors[idx, jdx]:.2e} |"
    print(row)

print("\nNotes:")
print("  - Power iteration amplifies dominant singular components")
print("  - Extra samples (oversampling) improves subspace capture")

Reconstruction Error
extra_samples |  iter=0  |  iter=1  |  iter=2  |  iter=3  |  iter=4  |  iter=8  |  iter=16  |
--------------+----------+----------+----------+----------+----------+----------+----------+
           24 | 5.41e-02 | 3.73e-02 | 3.72e-02 | 3.71e-02 | 3.71e-02 | 3.71e-02 | 3.71e-02 |
           18 | 5.73e-02 | 3.74e-02 | 3.72e-02 | 3.71e-02 | 3.71e-02 | 3.71e-02 | 3.71e-02 |
           12 | 6.18e-02 | 3.75e-02 | 3.72e-02 | 3.72e-02 | 3.71e-02 | 3.71e-02 | 3.71e-02 |
            6 | 6.82e-02 | 3.77e-02 | 3.72e-02 | 3.72e-02 | 3.72e-02 | 3.71e-02 | 3.71e-02 |
            3 | 7.35e-02 | 3.79e-02 | 3.74e-02 | 3.72e-02 | 3.72e-02 | 3.71e-02 | 3.71e-02 |


Singular Value Error
extra_samples |  iter=0  |  iter=1  |  iter=2  |  iter=3  |  iter=4  |  iter=8  |  iter=16  |
--------------+----------+----------+----------+----------+----------+----------+----------+
           24 | 5.52e-03 | 5.00e-04 | 1.41e-04 | 7.02e-05 | 3.85e-05 | 3.19e-06 | 8.37e-09 |
           18 | 6.59e-03